<img src="https://backend.burla.dev/static/welcome.svg" width="45%">

#### This notebook fires **1,000 HTTP requests** across 10 cloud workers with per-worker rate limiting, in about 3 minutes. <br/> It's only 4 steps — follow along!

### What is Burla?

Burla is the simplest way to run Python across hundreds or thousands of cloud machines.
One function — `remote_parallel_map` — fans your code out across isolated Docker containers, grows the cluster on demand, and streams results back. Missing packages on the workers? Burla auto-installs them.

No Dockerfiles, no cluster YAML, no queue. Just Python.  
Burla is free to try — this whole notebook runs on machines we spin up for you.

### What will this demo do?

You need to hit an API N times — enrich a user list, backfill, call an LLM — but the API has a rate limit. `asyncio` on one laptop tops out at a few thousand concurrent sockets. A naive parallel for-loop on 1,000 machines would hammer the API and get you banned.

The pattern: **exactly N workers running at once**, each doing a small share of the work, each respecting a local rate limit. Global throughput = per-worker rate × number of workers.

In this demo we hit **`https://jsonplaceholder.typicode.com`** (a free public test API) 1,000 times across **10 workers capped at 10 concurrent**. Each worker does ~1 req/sec locally, so global throughput is ~10 req/sec — comfortably under the API's limit.

The full-scale version in [`main.py`](https://github.com/Burla-Cloud/rate-limited-api-requests/blob/main/main.py) fires 2M requests across 1,000 workers at a global ~1,000 req/sec.

## 1)&nbsp; Boot some machines (10+ CPUs):
&nbsp;&nbsp;&nbsp;&nbsp;Sign in using the button below, then hit the **`⏻ Start`** button on your dashboard homepage.  
&nbsp;&nbsp;&nbsp;&nbsp;This will boot a small cluster in Google Cloud for you. Burla is free to try so this is on us!

&nbsp;&nbsp;
<a href="https://login.burla.dev">
    <img src="https://i.ibb.co/QFNncHcp/Clean-Shot-2026-03-19-at-18-13-07.jpg" width="28%">
</a>

## 2)&nbsp; Install the Python package:

In [ ]:
!uv pip install burla

## 3)&nbsp; Authorize this computer:

In [ ]:
!burla login

## 4)&nbsp; Fire 1,000 rate-limited API requests 🚀

Each worker gets a chunk of 100 post IDs. For each ID it fetches `GET /posts/{id}`, sleeps 1 second (per-worker rate limit), and retries on `429` with exponential backoff. `max_parallelism=10` makes sure we only have 10 workers live at once.

In [ ]:
import httpx  # noqa: F401  # top-level import -> Burla auto-installs httpx on workers
from burla import remote_parallel_map

# jsonplaceholder has post IDs 1..100; we repeat the range 10x to make 1,000 total requests
ids = list(range(1, 101)) * 10

CHUNK = 100
chunks = [ids[i : i + CHUNK] for i in range(0, len(ids), CHUNK)]
print(f"{len(ids):,} requests across {len(chunks)} tasks")


def fetch_chunk(post_ids: list[int]) -> list[dict]:
    import time
    import httpx

    out = []
    with httpx.Client(timeout=20.0) as client:
        for pid in post_ids:
            for attempt in range(5):
                r = client.get(f"https://jsonplaceholder.typicode.com/posts/{pid}")
                if r.status_code == 429:
                    wait = float(r.headers.get("Retry-After", 2 ** attempt))
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                data = r.json()
                out.append({"post_id": pid, "title_len": len(data["title"]), "body_len": len(data["body"])})
                break
            time.sleep(1.0)  # 1 req/sec per worker, capped at 10 workers = ~10 req/sec global
    return out


# 10 tasks, capped at 10 workers running in parallel
chunk_results = remote_parallel_map(
    fetch_chunk,
    chunks,
    func_cpu=1,
    func_ram=2,
    max_parallelism=10,
    grow=True,
)

rows = [r for chunk in chunk_results for r in chunk]
print(f"got {len(rows):,} responses")

### What just happened?

You just hit an API 1,000 times — with rate limiting that respects the API, retry on 429, and a configurable global concurrency cap — without writing a queue, a worker pool, or a retry library.

The `max_parallelism=10` argument is the key. You can set it to whatever the API allows and Burla will hold that many workers live at once, even as tasks finish and start.

### Inspect the results

Simple per-post stats — Burla just returned Python dicts, so we use pandas from here.

In [ ]:
import pandas as pd

results = pd.DataFrame(rows)
print(f"{len(results):,} rows, {results['post_id'].nunique()} unique posts")
print()
print("avg title length:", round(results["title_len"].mean(), 1))
print("avg body length:", round(results["body_len"].mean(), 1))
results.head()

### Try this next

- Scale up: crank `ids` to `100_000` requests and `max_parallelism` to `100`. Global rate stays fixed, total time scales linearly with queue depth.
- Swap the URL for your own API. Add headers, auth, POST bodies — it's just `httpx`.
- Tighten the rate: drop `time.sleep(1.0)` to `0.2` if your API allows 5 req/sec per client.
- Use `generator=True` so results stream back chunk-by-chunk (nice for long backfills).

## Thank you for trying Burla! ❤️

### Run the full 2M-request version

See [`main.py`](https://github.com/Burla-Cloud/rate-limited-api-requests/blob/main/main.py) in this repo — 2,000,000 requests across 1,000 workers at ~1,000 req/sec.

### Run this on your laptop 🧑‍💻

1. `pip install burla`
2. `burla login`
3. Run the same example code from above ☝️

Congrats! Your laptop now effectively has 1000+ CPUs, and 4000G of RAM &nbsp; : )

<br/>

### Any way we can help? Lets chat!

Schedule a meeting: https://cal.com/jakez  
Send me an email: jake@burla.dev